# Phase 3: Model Development (Classification Systems)
This notebook implements baseline and advanced machine learning classification models to support proactive hospital decision-making. We build two classifiers:
1. **Model A: Visit Risk Classification** - Predicts whether a hospital visit represents a Low, Medium, or High operational and clinical risk.
2. **Model B: Claim Outcome Classification** - Predicts whether an insurance claim will be Paid, Pending, or Rejected before submission.

Both models employ pipelines, time-based splits, hyperparameter tuning, and class imbalance mitigation strategies.

### Import Dependencies and Load Dataset

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

# Load modeling dataset
df = pd.read_csv("../data_outputs/model_table.csv")
print(f"Modeling dataset loaded successfully. Shape: {df.shape}")

Modeling dataset loaded successfully. Shape: (25000, 28)


### Preprocessing and Mapping Targets
We map categorical targets to integer labels:
- `risk_score` target: `'Low' -> 0, 'Medium' -> 1, 'High' -> 2`
- `claim_status` target: `'Paid' -> 0, 'Pending' -> 1, 'Rejected' -> 2`

In [2]:
# Map target variables
risk_map = {'Low': 0, 'Medium': 1, 'High': 2}
claim_map = {'Paid': 0, 'Pending': 1, 'Rejected': 2}

# Filter out rows where target is null
df = df[df['risk_score'].notnull() & df['claim_status'].notnull()].copy()

df['risk_score_encoded'] = df['risk_score'].map(risk_map)
df['claim_status_encoded'] = df['claim_status'].map(claim_map)

print("Target distributions:")
print("Risk Score:", df['risk_score'].value_counts(normalize=True).round(3).to_dict())
print("Claim Status:", df['claim_status'].value_counts(normalize=True).round(3).to_dict())

Target distributions:
Risk Score: {'Low': 0.499, 'Medium': 0.3, 'High': 0.201}
Claim Status: {'Paid': 0.598, 'Pending': 0.251, 'Rejected': 0.152}


### Train / Test Split Strategy
We split based on the predefined `split` column (80% training on earlier dates, 20% testing on later dates) which prevents data leakage.

In [3]:
train_df = df[df['split'] == 'train'].copy()
test_df = df[df['split'] == 'test'].copy()

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

Train size: 20000, Test size: 5000


## Model A: Visit Risk Classification (Low, Medium, High)
**Feature Selection Logical Boundary:** Assessed at admission, so we only use demographic, temporal, and patient historical features. We **must not** use billing details (like `billed_amount` or `claim_status`).

In [4]:
features_A = [
    'age', 'gender', 'city', 'chronic_flag', 
    'length_of_stay_hours', 'department', 'visit_type',
    'days_since_registration', 'visit_month', 'visit_day_of_week',
    'visit_frequency', 'avg_length_of_stay_patient'
]

X_train_A = train_df[features_A].copy()
y_train_A = train_df['risk_score_encoded'].values

X_test_A = test_df[features_A].copy()
y_test_A = test_df['risk_score_encoded'].values

# Define preprocessor
num_features_A = ['age', 'length_of_stay_hours', 'days_since_registration', 'visit_month', 'visit_day_of_week', 'visit_frequency', 'avg_length_of_stay_patient']
cat_features_A = ['gender', 'city', 'department', 'visit_type']

preprocessor_A = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features_A),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features_A)
    ]
)
print("Preprocessor A defined.")

Preprocessor A defined.


### Model A Baseline: Logistic Regression

In [5]:
pipeline_lr_A = Pipeline([
    ('preprocessor', preprocessor_A),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline_lr_A.fit(X_train_A, y_train_A)
preds_lr_A = pipeline_lr_A.predict(X_test_A)
print("Model A Baseline Accuracy:", accuracy_score(y_test_A, preds_lr_A))
print(classification_report(y_test_A, preds_lr_A, target_names=['Low', 'Medium', 'High']))

Model A Baseline Accuracy: 0.4952
              precision    recall  f1-score   support

         Low       0.50      1.00      0.66      2478
      Medium       0.29      0.00      0.00      1500
        High       0.00      0.00      0.00      1022

    accuracy                           0.50      5000
   macro avg       0.26      0.33      0.22      5000
weighted avg       0.33      0.50      0.33      5000



D:\Projects\IITM\Capstone Project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Projects\IITM\Capstone Project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Projects\IITM\Capstone Project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

### Model A Advanced: Random Forest & Hyperparameter Tuning

In [6]:
pipeline_rf_A = Pipeline([
    ('preprocessor', preprocessor_A),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Define search space
param_grid_A = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [10, 15, None],
    'classifier__min_samples_split': [2, 5]
}

grid_search_A = GridSearchCV(pipeline_rf_A, param_grid_A, cv=3, scoring='accuracy', n_jobs=-1)
grid_search_A.fit(X_train_A, y_train_A)
best_model_A = grid_search_A.best_estimator_

print("Best params A:", grid_search_A.best_params_)
preds_rf_A = best_model_A.predict(X_test_A)
print("Model A Advanced Accuracy:", accuracy_score(y_test_A, preds_rf_A))
print(classification_report(y_test_A, preds_rf_A, target_names=['Low', 'Medium', 'High']))

Best params A: {'classifier__max_depth': 10, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100}
Model A Advanced Accuracy: 0.4952
              precision    recall  f1-score   support

         Low       0.50      0.95      0.66      2478
      Medium       0.38      0.08      0.13      1500
        High       0.00      0.00      0.00      1022

    accuracy                           0.50      5000
   macro avg       0.29      0.34      0.26      5000
weighted avg       0.36      0.50      0.37      5000



## Model B: Claim Outcome Classification (Paid, Pending, Rejected)
**Feature Selection Logical Boundary:** Predicts whether an insurance claim is Paid, Pending, or Rejected. We can use `risk_score` (Model A target) and `billed_amount` (known at billing time), but we **must not** use `approved_amount` or `payment_days` to prevent target leakage.

In [7]:
features_B = [
    'age', 'gender', 'city', 'chronic_flag', 
    'length_of_stay_hours', 'department', 'visit_type',
    'days_since_registration', 'visit_month', 'visit_day_of_week',
    'visit_frequency', 'avg_length_of_stay_patient',
    'risk_score', 'billed_amount', 'insurance_provider', 
    'provider_rejection_rate', 'revenue_realization_rate_dept'
]

X_train_B = train_df[features_B].copy()
y_train_B = train_df['claim_status_encoded'].values

X_test_B = test_df[features_B].copy()
y_test_B = test_df['claim_status_encoded'].values

# Preprocessing setup
num_features_B = [
    'age', 'length_of_stay_hours', 'days_since_registration', 
    'visit_month', 'visit_day_of_week', 'visit_frequency', 
    'avg_length_of_stay_patient', 'billed_amount', 
    'provider_rejection_rate', 'revenue_realization_rate_dept'
]
cat_features_B = ['gender', 'city', 'department', 'visit_type', 'risk_score', 'insurance_provider']

preprocessor_B = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features_B),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features_B)
    ]
)
print("Preprocessor B defined.")

Preprocessor B defined.


### Model B Baseline: Logistic Regression (Class Imbalance Mitigated)

In [8]:
# Using class_weight='balanced' to handle the minority categories
pipeline_lr_B = Pipeline([
    ('preprocessor', preprocessor_B),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

pipeline_lr_B.fit(X_train_B, y_train_B)
preds_lr_B = pipeline_lr_B.predict(X_test_B)
print("Model B Baseline Accuracy:", accuracy_score(y_test_B, preds_lr_B))
print(classification_report(y_test_B, preds_lr_B, target_names=['Paid', 'Pending', 'Rejected']))

Model B Baseline Accuracy: 0.321
              precision    recall  f1-score   support

        Paid       0.65      0.30      0.41      2994
     Pending       0.25      0.28      0.26      1276
    Rejected       0.17      0.49      0.25       730

    accuracy                           0.32      5000
   macro avg       0.35      0.36      0.31      5000
weighted avg       0.47      0.32      0.35      5000



### Model B Advanced: LightGBM & Hyperparameter Tuning

In [9]:
pipeline_lgb_B = Pipeline([
    ('preprocessor', preprocessor_B),
    ('classifier', lgb.LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1))
])

param_grid_B = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 8, 10],
    'classifier__learning_rate': [0.05, 0.1]
}

grid_search_B = GridSearchCV(pipeline_lgb_B, param_grid_B, cv=3, scoring='accuracy', n_jobs=-1)
grid_search_B.fit(X_train_B, y_train_B)
best_model_B = grid_search_B.best_estimator_

print("Best params B:", grid_search_B.best_params_)
preds_lgb_B = best_model_B.predict(X_test_B)
print("Model B Advanced Accuracy:", accuracy_score(y_test_B, preds_lgb_B))
print(classification_report(y_test_B, preds_lgb_B, target_names=['Paid', 'Pending', 'Rejected']))

Best params B: {'classifier__learning_rate': 0.05, 'classifier__max_depth': 10, 'classifier__n_estimators': 50}
Model B Advanced Accuracy: 0.4094
              precision    recall  f1-score   support

        Paid       0.67      0.44      0.53      2994
     Pending       0.28      0.19      0.23      1276
    Rejected       0.22      0.66      0.34       730

    accuracy                           0.41      5000
   macro avg       0.39      0.43      0.37      5000
weighted avg       0.51      0.41      0.43      5000



D:\Projects\IITM\Capstone Project\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### Serialize and Save Model Artifacts
We serialize the complete fitted pipeline objects (`best_model_A` and `best_model_B`). These include both the preprocessing transformers and the trained classifiers, allowing them to accept raw input dataframes directly during deployment, satisfying MLOps requirements.

In [10]:
# Create models directory
os.makedirs("../models", exist_ok=True)

# Save Model A
joblib.dump(best_model_A, "../models/risk_model.pkl")
print("Saved Model A to 'models/risk_model.pkl'!")

# Save Model B
joblib.dump(best_model_B, "../models/claim_model.pkl")
print("Saved Model B to 'models/claim_model.pkl'!")

Saved Model A to 'models/risk_model.pkl'!
Saved Model B to 'models/claim_model.pkl'!
